# wav2vec 2.0 + KenLM decoding (Colab)

KenLM does not install easily on Windows, so the LM part of the wav2vec 2.0 system runs here.

1. Clone the repo and install pinned packages
2. Build a 3-gram LM from **train-split text only** (`build_lm.sh`)
3. Choose `alpha` / `beta` on **dev** (`tune_lm.py`)
4. Decode **test once** with the chosen values, greedy and with the LM (`transcribe.py`)

The fine-tuned model is loaded from the Hugging Face Hub, so nothing is uploaded by hand.
Use a GPU runtime (Runtime → Change runtime type → T4 GPU). RTF measured here is on Colab hardware, which the timing files record; compare it only with rows from the same hardware.

In [ ]:
%cd /content
![ -d urdu-asr-kathbath ] || git clone -q --depth 1 https://github.com/r4fo/urdu-asr-kathbath.git
%cd /content/urdu-asr-kathbath
# Colab runs Python 3.13, where kenlm and pyctcdecode do not build. So the LM steps run in a Python 3.11
# environment with the same pinned packages as requirements.txt (Python 3.11 locally too), plus the LM decoder.
!apt-get -qq update > /dev/null; apt-get -qq install -y build-essential cmake libboost-program-options-dev libboost-system-dev libboost-thread-dev libboost-test-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev > /dev/null
!pip install -q uv
!uv venv -q -p 3.11 /content/py311
!uv pip install -q -p /content/py311/bin/python -r requirements.txt pyctcdecode==0.5.0 kenlm==0.2.0
import subprocess
PY = "/content/py311/bin/python"
# stops "Run all" here if anything is missing
subprocess.run([PY, "-c", "import torch, kenlm, pyctcdecode; assert torch.cuda.is_available(), 'no GPU'; print('kenlm and pyctcdecode OK, GPU:', torch.cuda.get_device_name(0))"], check=True)

In [ ]:
# Fine-tuned model on the Hub, pinned to an exact commit hash
from huggingface_hub import HfApi
MODEL = "r4fo/wav2vec2-xls-r-300m-urdu-kathbath"
REVISION = HfApi().model_info(MODEL).sha
print(MODEL, REVISION)  # this hash goes in the README

In [ ]:
!bash models/wav2vec2_finetune/build_lm.sh

In [ ]:
import os
assert os.path.exists("lm/train_3gram.arpa"), "LM was not built: check the output of the cell above"
!{PY} models/wav2vec2_finetune/tune_lm.py --model $MODEL --revision $REVISION --manifest data/splits/dev.csv --lm lm/train_3gram.arpa

In [ ]:
import pandas as pd
best = pd.read_csv("results/lm_tuning_dev.csv").iloc[0]
ALPHA, BETA = best.alpha, best.beta
print(f"Chosen on dev: alpha={ALPHA}, beta={BETA}, dev WER={best.wer:.2%}")

In [ ]:
assert os.path.exists("results/lm_tuning_dev.csv"), "dev tuning did not finish: do not run test"
# Test, once, with the values chosen on dev
!{PY} models/wav2vec2_finetune/transcribe.py --model $MODEL --revision $REVISION --manifest data/splits/test.csv --out results/predictions/w2v2_xlsr_ft_kenlm.csv --system w2v2_xlsr_ft_kenlm --lm lm/train_3gram.arpa --alpha $ALPHA --beta $BETA
# Greedy on the same Colab GPU, only for a like-for-like speed comparison with the LM row
!{PY} models/wav2vec2_finetune/transcribe.py --model $MODEL --revision $REVISION --manifest data/splits/test.csv --out results/predictions/w2v2_xlsr_ft_greedy_colab.csv --system w2v2_xlsr_ft_greedy_colab

In [ ]:
!{PY} src/evaluate.py --split data/splits/test.csv --pred results/predictions/w2v2_xlsr_ft_kenlm.csv results/predictions/w2v2_xlsr_ft_greedy_colab.csv

In [ ]:
# Download these and commit them from your own machine
!zip -q lm_outputs.zip lm/train_3gram.arpa results/lm_tuning_dev.csv results/predictions/w2v2_xlsr_ft_kenlm.csv results/predictions/w2v2_xlsr_ft_greedy_colab.csv results/timing/w2v2_xlsr_ft_kenlm.csv results/timing/w2v2_xlsr_ft_greedy_colab.csv
from google.colab import files
files.download("lm_outputs.zip")